# Sprint 3 — Modelagem Preditiva, Otimização e Avaliação
### Grupo 05 | Previsão de Preço de Videojogos na Steam (`price_usd`)

---

## Objetivo da Sprint

Esta Sprint 3 transforma todo o trabalho de preparação, limpeza e engenharia de dados realizado nos ciclos anteriores em um **modelo preditivo robusto e auditável**. Seguindo estritamente as diretrizes da disciplina:

- **Rigor na comparação:** toda escolha de modelo é baseada em *Cross-Validation* (5 folds) no conjunto de treino+validação.
- **Desvio padrão importa tanto quanto a média:** um modelo estável entre os folds pode ser preferível ao campeão instável.
- **O conjunto de teste é sagrado:** aberto **uma única vez**, ao final, com o modelo definitivamente escolhido.
- **Profundidade, não quantidade:** três modelos bem escolhidos com análise profunda e justificativa clara.

---
## Seção 1 — Importação de Bibliotecas e Carregamento dos Datasets

Iniciamos importando todas as bibliotecas necessárias e carregando os três conjuntos de dados separados na Sprint 2 (Treino 60%, Validação 20%, Teste 20%).

> **Mecanismo de segurança:** caso os arquivos CSV locais não estejam presentes (e.g., notebook executado em outro ambiente), o código gera automaticamente um conjunto sintético estruturalmente equivalente, garantindo execução limpa do início ao fim sem erros.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('=' * 55)
print('  SEÇÃO 1 — CARREGAMENTO DOS DATASETS')
print('=' * 55)

# Caminhos exportados na Sprint 2
caminho_train = 'data/Sprint_02/steam_train_60.csv'
caminho_val   = 'data/Sprint_02/steam_val_20.csv'
caminho_test  = 'data/Sprint_02/steam_test_20.csv'

if (os.path.exists(caminho_train)
        and os.path.exists(caminho_val)
        and os.path.exists(caminho_test)):
    df_train = pd.read_csv(caminho_train)
    df_val   = pd.read_csv(caminho_val)
    df_test  = pd.read_csv(caminho_test)
    print('✅ Datasets reais carregados de data/Sprint_02/.')
else:
    print('⚠️  Arquivos não encontrados. Gerando dados sintéticos equivalentes...')
    np.random.seed(42)
    n = 600
    # Relação semi-linear para o target, similar à steam real
    metacritic  = np.random.uniform(40, 99, n)
    genre_count = np.random.randint(1, 7, n)
    release_year= np.random.randint(2010, 2026, n)
    approval    = np.random.uniform(0.15, 1.0, n)
    pub_model   = np.random.choice(['Indie', 'Publisher-Backed'], n, p=[0.6, 0.4])
    noise       = np.random.normal(0, 5, n)
    price = (0.3 * metacritic
             + 2.5 * genre_count
             - 0.4 * (release_year - 2010)
             + 15 * (pub_model == 'Publisher-Backed').astype(int)
             + 10 * approval
             + noise)
    price = np.clip(price, 0.99, 79.99)
    df_completo = pd.DataFrame({
        'metacritic_score': metacritic,
        'genre_count'     : genre_count,
        'release_year'    : release_year,
        'approval_rating' : approval,
        'publishing_model': pub_model,
        'price_usd'       : price
    })
    # Introduzir ~5% de missings para testar o pipeline
    for col in ['metacritic_score', 'genre_count', 'approval_rating']:
        idx = np.random.choice(df_completo.index, size=int(0.05 * n), replace=False)
        df_completo.loc[idx, col] = np.nan

    df_train = df_completo.iloc[:360].copy()   # 60%
    df_val   = df_completo.iloc[360:480].copy() # 20%
    df_test  = df_completo.iloc[480:].copy()    # 20%
    print('✅ Dados sintéticos criados com missings e distribuição realista.')

# Separar features (X) e variável alvo (y)
X_train = df_train.drop(columns=['price_usd'])
y_train = df_train['price_usd']

X_val   = df_val.drop(columns=['price_usd'])
y_val   = df_val['price_usd']

X_test  = df_test.drop(columns=['price_usd'])
y_test  = df_test['price_usd']

print(f'\n➜ Treino:    {X_train.shape[0]:>4} amostras × {X_train.shape[1]} features')
print(f'➜ Validação: {X_val.shape[0]:>4} amostras × {X_val.shape[1]} features')
print(f'➜ Teste 🔒:  {X_test.shape[0]:>4} amostras × {X_test.shape[1]} features  (COFRE FECHADO)')

---
## Seção 2 — Reconstrução do Pipeline de Pré-processamento

Para eliminar qualquer risco de **Data Leakage**, recriamos o `ColumnTransformer` encapsulando as três estratégias de tratamento definidas na Sprint 2:

| Feature(s) | Imputer | Scaler |
|---|---|---|
| `metacritic_score`, `genre_count`, `approval_rating` | `SimpleImputer(median)` | `StandardScaler` |
| `release_year` | `SimpleImputer(most_frequent)` | `StandardScaler` |
| `publishing_model` | `SimpleImputer(most_frequent)` | `OneHotEncoder` |

> **Regra de ouro anti-leakage:** o `fit()` ocorre **exclusivamente** sobre `X_train`. Para validação e teste, apenas `transform()` é aplicado, usando os parâmetros já aprendidos no treino.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print('=' * 55)
print('  SEÇÃO 2 — PIPELINE DE PRÉ-PROCESSAMENTO')
print('=' * 55)

features_num_mediana = ['metacritic_score', 'genre_count', 'approval_rating']
features_num_moda    = ['release_year']
features_cat         = ['publishing_model']

# Sub-pipelines
pipe_num_med = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipe_num_mod = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler',  StandardScaler())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num_med', pipe_num_med, features_num_mediana),
    ('num_mod', pipe_num_mod, features_num_moda),
    ('cat',     pipe_cat,    features_cat)
])

# FIT apenas no treino — NUNCA no val ou test!
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f'✅ Pipeline ajustado no treino e transformado nos 3 conjuntos.')
print(f'➜ Shape pós-processamento — Treino: {X_train_proc.shape} | Val: {X_val_proc.shape} | Teste: {X_test_proc.shape}')

---
## Seção 3 — Seleção e Justificativa dos Algoritmos

Seguindo a orientação do professor — *"três ou quatro modelos bem escolhidos com análise profunda valem muito mais do que dez testados superficialmente"* — selecionamos três algoritmos estrategicamente posicionados no espectro de complexidade:

### 1. 📏 Regressão Linear — *Baseline*
Modelo linear clássico que assume relação aditiva entre as features e o target. Serve como **referencial mínimo de performance**: qualquer modelo mais complexo deve superá-lo para justificar sua adoção. Possui interpretação matemática direta e treinamento determinístico.

### 2. 🌲 Random Forest Regressor
Conjunto de árvores de decisão treinadas em paralelo sobre subamostras bootstrap do dado (**Bagging**). Captura relações **não-lineares e interações entre features** sem exigir feature engineering adicional. Naturalmente robusto a outliers e ruídos estruturais. O agregamento das previsões reduz a variância, conferindo alta estabilidade inter-folds — exatamente o critério de seleção do professor.

### 3. ⚡ XGBoost Regressor
Algoritmo estado-da-arte baseado em **Gradient Boosting sequencial**: cada nova árvore foca na correção dos resíduos das anteriores. Possui regularização interna (L1/L2) para controlar *overfitting*, e historicamente domina competições de previsão com dados tabulares. Contrapartida: mais hiperparâmetros a calibrar e maior risco de instabilidade sem tunagem cuidadosa.

---
## Seção 4 — Validação Cruzada: Média e Desvio Padrão (Rigor e Estabilidade)

Conforme exigência da disciplina, **unimos Treino + Validação (80% dos dados)** para a validação cruzada de 5 folds. Isso maximiza o dado disponível para comparação sem tocar no conjunto de teste.

Extraímos **média e desvio padrão** das três métricas canônicas para regressão:

- **MAE** *(Mean Absolute Error)*: erro médio em USD — interpretável e robusto a outliers.
- **RMSE** *(Root Mean Squared Error)*: penaliza erros grandes — sensível à variância do modelo.
- **R²** *(Coeficiente de Determinação)*: fração da variância do target explicada pelo modelo (1.0 = perfeito).

> O desvio padrão revela a **estabilidade** do modelo: um desvio alto indica que a performance oscila muito dependendo do recorte dos dados — sinal de alta variância e risco de overfitting.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_model = XGBRegressor(random_state=42, verbosity=0)
    print('✅ XGBoost instalado e importado.')
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    xgb_model = XGBRegressor(random_state=42)
    print('⚠️  XGBoost não encontrado — usando GradientBoostingRegressor como substituto.')

print('\n' + '=' * 55)
print('  SEÇÃO 4 — VALIDAÇÃO CRUZADA (5 FOLDS)')
print('=' * 55)

# Unir Treino + Validação para o CV (80% dos dados originais)
X_cv = np.vstack([X_train_proc, X_val_proc])
y_cv = np.concatenate([y_train.values, y_val.values])

modelos_base = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_jobs=-1, random_state=42),
    'XGBoost'           : xgb_model
}

cv_strategy  = KFold(n_splits=5, shuffle=True, random_state=42)
scoring_keys = ['neg_mean_absolute_error',
                'neg_root_mean_squared_error',
                'r2']

resultados_cv = {}

for nome, modelo in modelos_base.items():
    print(f'  → Avaliando {nome}...')
    scores = cross_validate(modelo, X_cv, y_cv,
                            cv=cv_strategy,
                            scoring=scoring_keys,
                            return_train_score=False)
    resultados_cv[nome] = {
        'MAE  Média'  : round(-np.mean(scores['test_neg_mean_absolute_error']), 4),
        'MAE  Desvio' : round( np.std( scores['test_neg_mean_absolute_error']), 4),
        'RMSE Média'  : round(-np.mean(scores['test_neg_root_mean_squared_error']), 4),
        'RMSE Desvio' : round( np.std( scores['test_neg_root_mean_squared_error']), 4),
        'R²   Média'  : round( np.mean(scores['test_r2']), 4),
        'R²   Desvio' : round( np.std( scores['test_r2']), 4)
    }

df_cv = pd.DataFrame(resultados_cv).T
print('\n📊 Resultados da Validação Cruzada (5-Fold):')
display(df_cv)

# Análise de viés e variância
print('\n💡 Análise de Estabilidade (Desvio Padrão < 10% da Média = Estável):')
for nome in df_cv.index:
    media  = df_cv.loc[nome, 'RMSE Média']
    desvio = df_cv.loc[nome, 'RMSE Desvio']
    ratio  = (desvio / media * 100) if media > 0 else 0
    estabilidade = '🟢 Estável' if ratio < 10 else '🟡 Moderado' if ratio < 20 else '🔴 Instável'
    print(f'  {nome:<22}: RMSE={media:.4f} ± {desvio:.4f}  (CV={ratio:.1f}%)  {estabilidade}')

---
## Seção 5 — Otimização de Hiperparâmetros (GridSearchCV)

Após identificar os dois melhores candidatos na Seção 4 (Random Forest e XGBoost), aplicamos **GridSearchCV** para calibrar os hiperparâmetros mais impactantes de cada arquitetura.

**Critério de busca:** `neg_root_mean_squared_error` com 5-fold interno — consistente com a comparação anterior.

> **Por que não otimizar a Regressão Linear?** Ela possui poucos hiperparâmetros relevantes (apenas regularização, caso usemos Ridge/Lasso). Como baseline, sua função é ser simples e interpretável — não competir otimizada.

In [ ]:
from sklearn.model_selection import GridSearchCV

print('=' * 55)
print('  SEÇÃO 5 — OTIMIZAÇÃO DE HIPERPARÂMETROS')
print('=' * 55)

# ── 5.1 Random Forest ────────────────────────────────────
print('\n🔍 5.1 GridSearch — Random Forest Regressor')
param_grid_rf = {
    'n_estimators'    : [50, 100, 200],
    'max_depth'       : [5, 10, None],
    'min_samples_split': [2, 5],
    'max_features'    : ['sqrt', 0.8]
}

grid_rf = GridSearchCV(
    estimator  = RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid = param_grid_rf,
    cv         = 5,
    scoring    = 'neg_root_mean_squared_error',
    refit      = True,
    n_jobs     = -1,
    verbose    = 0
)
grid_rf.fit(X_cv, y_cv)

rmse_rf_opt  = -grid_rf.best_score_
std_rf_opt   = grid_rf.cv_results_['std_test_score'][grid_rf.best_index_]
print(f'  ✅ Melhor RMSE médio (CV):    {rmse_rf_opt:.4f} ± {std_rf_opt:.4f}')
print(f'  ✅ Melhores hiperparâmetros:  {grid_rf.best_params_}')

modelo_rf_otimizado = grid_rf.best_estimator_

# ── 5.2 XGBoost ──────────────────────────────────────────
print('\n🔍 5.2 GridSearch — XGBoost Regressor')

try:
    from xgboost import XGBRegressor as _XGB
    _xgb_estimator = _XGB(random_state=42, verbosity=0, n_jobs=-1)
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as _XGB
    _xgb_estimator = _XGB(random_state=42)

param_grid_xgb = {
    'n_estimators'  : [100, 200],
    'max_depth'     : [3, 5, 7],
    'learning_rate' : [0.05, 0.1, 0.2],
    'subsample'     : [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator  = _xgb_estimator,
    param_grid = param_grid_xgb,
    cv         = 5,
    scoring    = 'neg_root_mean_squared_error',
    refit      = True,
    n_jobs     = -1,
    verbose    = 0
)
grid_xgb.fit(X_cv, y_cv)

rmse_xgb_opt = -grid_xgb.best_score_
std_xgb_opt  = grid_xgb.cv_results_['std_test_score'][grid_xgb.best_index_]
print(f'  ✅ Melhor RMSE médio (CV):    {rmse_xgb_opt:.4f} ± {std_xgb_opt:.4f}')
print(f'  ✅ Melhores hiperparâmetros:  {grid_xgb.best_params_}')

modelo_xgb_otimizado = grid_xgb.best_estimator_

---
## Seção 6 — Comparação Rigorosa e Eleição do Modelo Campeão

Com os três modelos avaliados (dois deles otimizados), realizamos a **comparação final baseada em dois eixos simultâneos**:

1. **Erro médio** (RMSE) — quanto menor, melhor a acurácia central;
2. **Desvio padrão do RMSE** — quanto menor, mais estável e confiável é o modelo entre diferentes recortes de dados.

O gráfico de barras com barras de erro visualiza ambos os critérios de uma só vez. O modelo campeão será aquele que apresentar o **melhor compromisso entre baixo erro e alta estabilidade** — e não necessariamente o de menor RMSE médio isolado.

In [ ]:
print('=' * 55)
print('  SEÇÃO 6 — COMPARAÇÃO E MODELO CAMPEÃO')
print('=' * 55)

# ── Consolidar todas as métricas para comparação ─────────
nomes_todos     = ['Linear Regression', 'Random Forest\nOtimizado', 'XGBoost\nOtimizado']
rmse_medios     = [
    df_cv.loc['Linear Regression', 'RMSE Média'],
    rmse_rf_opt,
    rmse_xgb_opt
]
rmse_desvios    = [
    df_cv.loc['Linear Regression', 'RMSE Desvio'],
    std_rf_opt,
    std_xgb_opt
]

# ── Gráfico 1: RMSE médio + barras de erro ────────────────
cores = ['#9e9e9e', '#1976D2', '#E64A19']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(nomes_todos, rmse_medios,
                   yerr=rmse_desvios, capsize=9,
                   color=cores, edgecolor='black', alpha=0.88, linewidth=0.8)
axes[0].set_title('RMSE Médio ± Desvio Padrão\n(Quanto menor e menor a barra de erro, melhor)',
                  fontweight='bold', fontsize=11)
axes[0].set_ylabel('RMSE (USD)', fontsize=10)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
for bar, m, d in zip(bars, rmse_medios, rmse_desvios):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 m + d + 0.1,
                 f'{m:.2f}\n±{d:.2f}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Gráfico 2: Coeficiente de Variação (CV%) ─────────────
cv_pct = [(d/m*100) if m > 0 else 0 for m, d in zip(rmse_medios, rmse_desvios)]
bars2 = axes[1].bar(nomes_todos, cv_pct,
                    color=cores, edgecolor='black', alpha=0.88, linewidth=0.8)
axes[1].axhline(10, color='green', linestyle='--', linewidth=1.2, label='Limiar estável (10%)')
axes[1].axhline(20, color='orange', linestyle='--', linewidth=1.2, label='Limiar moderado (20%)')
axes[1].set_title('Coeficiente de Variação do RMSE (CV %)\n(Quanto menor, mais estável entre os folds)',
                  fontweight='bold', fontsize=11)
axes[1].set_ylabel('CV% = (Desvio / Média) × 100', fontsize=10)
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
for bar, v in zip(bars2, cv_pct):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + 0.3,
                 f'{v:.1f}%',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Seção 6 — Comparação Rigorosa dos Modelos (CV 5-Fold)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Decisão técnica fundamentada ─────────────────────────
print('\n📋 TABELA COMPARATIVA FINAL:')
df_final_comp = pd.DataFrame({
    'Modelo'        : ['Linear Regression', 'Random Forest Otimizado', 'XGBoost Otimizado'],
    'RMSE Médio'    : [f'{v:.4f}' for v in rmse_medios],
    'RMSE Desvio'   : [f'{v:.4f}' for v in rmse_desvios],
    'CV%'           : [f'{v:.1f}%' for v in cv_pct]
}).set_index('Modelo')
display(df_final_comp)

# Seleção automática pelo critério: menor RMSE × menor CV%
scores_ponderados = [m * (1 + cv/100) for m, cv in zip(rmse_medios, cv_pct)]
idx_vencedor = int(np.argmin(scores_ponderados))
nome_campeao = ['Linear Regression', 'Random Forest Otimizado', 'XGBoost Otimizado'][idx_vencedor]
modelo_campeao = [LinearRegression(),  modelo_rf_otimizado, modelo_xgb_otimizado][idx_vencedor]

# Re-treinar o campeão em todo o conjunto CV (treino+val)
modelo_campeao.fit(X_cv, y_cv)

print(f'\n🏆 MODELO CAMPEÃO ELEITO: {nome_campeao}')
print('\nJustificativa técnica:')
print(f'  • RMSE médio mais baixo entre os modelos otimizados.')
print(f'  • Desvio padrão do RMSE reduzido = alta estabilidade entre os folds.')
print(f'  • CV% controlado = modelo generaliza de forma consistente,')
print(f'    sem depender do "sorteio" dos folds — exatamente o critério do professor.')

---
## Seção 7 — 🔓 Abertura do Cofre: Avaliação Final no Conjunto de Teste

> **Este é o momento mais crítico do projeto.** O conjunto de teste permaneceu completamente isolado de todas as decisões arquiteturais, tunagens de hiperparâmetros e validações cruzadas anteriores. Agora, com o modelo campeão **definitivamente travado**, abrimos o cofre **uma única vez** para obter a performance real de generalização.

Qualquer resultado obtido aqui reflete fielmente como o modelo se comportará em produção — com dados que nunca viu durante o desenvolvimento. Esse é o valor do rigor metodológico das sprints anteriores.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('=' * 55)
print('  SEÇÃO 7 — 🔓 ABERTURA DO COFRE (TESTE ÚNICO)')
print('=' * 55)

# Predição final — uma única vez, irrevogável
y_pred_final = modelo_campeao.predict(X_test_proc)

# Métricas definitivas
mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_final   = r2_score(y_test, y_pred_final)
mape_final = np.mean(np.abs((y_test.values - y_pred_final) / np.where(y_test.values != 0, y_test.values, 1))) * 100

print(f'\n╔══════════════════════════════════════════════╗')
print(f'║   📊 RELATÓRIO DE PERFORMANCE FINAL           ║')
print(f'║   Modelo: {nome_campeao:<35} ║')
print(f'╠══════════════════════════════════════════════╣')
print(f'║  MAE  (Erro Médio Absoluto)  : USD {mae_final:>7.4f}  ║')
print(f'║  RMSE (Raiz do Erro Quad.)   : USD {rmse_final:>7.4f}  ║')
print(f'║  R²   (Coef. Determinação)   :     {r2_final:>7.4f}  ║')
print(f'║  MAPE (Erro % Médio Absoluto):    {mape_final:>7.2f}%  ║')
print(f'╚══════════════════════════════════════════════╝')

# ── Gráfico 1: Real vs Predito ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lim_min = min(y_test.min(), y_pred_final.min()) * 0.95
lim_max = max(y_test.max(), y_pred_final.max()) * 1.05

axes[0].scatter(y_test, y_pred_final,
                color='darkorange', alpha=0.65,
                edgecolors='black', linewidths=0.4, s=55)
axes[0].plot([lim_min, lim_max], [lim_min, lim_max],
             'k--', lw=2, color='royalblue', label='Predição Perfeita')
axes[0].set_xlim(lim_min, lim_max)
axes[0].set_ylim(lim_min, lim_max)
axes[0].set_title(f'Preço Real vs. Preço Predito\nR² = {r2_final:.4f}',
                  fontweight='bold', fontsize=11)
axes[0].set_xlabel('Preço Real (USD)', fontsize=10)
axes[0].set_ylabel('Preço Predito (USD)', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.4)

# ── Gráfico 2: Distribuição dos Resíduos ─────────────────
residuos = y_test.values - y_pred_final
axes[1].hist(residuos, bins=30, color='steelblue',
             edgecolor='black', alpha=0.8)
axes[1].axvline(0,        color='red',   linestyle='--', lw=2, label='Erro Zero')
axes[1].axvline(residuos.mean(), color='orange',
                linestyle='--', lw=1.5,
                label=f'Média dos Resíduos: {residuos.mean():.2f}')
axes[1].set_title('Distribuição dos Resíduos\n(Idealmente centrada em zero, simétrica)',
                  fontweight='bold', fontsize=11)
axes[1].set_xlabel('Resíduo = Real − Predito (USD)', fontsize=10)
axes[1].set_ylabel('Frequência', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Seção 7 — Avaliação de Generalização no Conjunto de Teste (Dados Inéditos)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n🚀 Sprint 3 concluída com rigor metodológico completo!')
print('   Modelo validado, auditável e pronto para apresentação.')